In [1]:
import cv2
from ultralytics import YOLO

# 1. Load your CPU-optimized OpenVINO model (or fallback to "models/best.pt")
model = YOLO("../models/best.pt")
model.export(format="openvino")

# 2. Open a video file (or use 0 for your laptop webcam)
cap = cv2.VideoCapture("test_video.mp4")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # 3. Run YOLO detection + ByteTrack persistence across frames
    results = model.track(source=frame, persist=True, tracker="bytetrack.yaml", device="cpu", verbose=False)

    # 4. Extract boxes and tracking IDs
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)
        class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

        for box, track_id, cls_id in zip(boxes, track_ids, class_ids):
            x1, y1, x2, y2 = box
            label = f"ID:{track_id} {model.names[cls_id]}"

            # Draw bounding rectangle and center point (useful for VisionNav spatial logic)
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("VisionNav - ByteTrack Video", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


Ultralytics 8.4.216 🚀 Python-3.14.4 torch-2.15.0+cu132 CPU (Intel Core i5-8265U 1.60GHz)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from '../models/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.3 MB)

OpenVINO: starting export with openvino 2026.4.1-22982-e213a147257-releases/2026/4...


/run/media/chethan/Disk/Transfer/vision/Vision_L/.venv/lib/python3.14/site-packages/torch/jit/_trace.py:1000: FutureWarning: `torch.jit.trace` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/run/media/chethan/Disk/Transfer/vision/Vision_L/.venv/lib/python3.14/site-packages/torch/jit/_trace.py:1139: FutureWarning: `torch.jit.trace_method` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


OpenVINO: export success ✅ 4.2s, saved as '../models/best_openvino_model/' (10.4 MB)

Export complete (6.9s)
Results saved to /run/media/chethan/Disk/Transfer/vision/Vision_L/models/best_openvino_model
Predict:         yolo predict task=detect model=../models/best_openvino_model/ imgsz=640 
Validate:        yolo val task=detect model=../models/best_openvino_model/ imgsz=640  
Visualize:       https://netron.app
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.14.4 environment at: /run/media/chethan/Disk/Transfer/vision/Vision_L/.venv
Resolved 2 packages in 2.94s
 Downloaded lap
Prepared 1 package in 9.25s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 245ms
 + lap==0.5.14

requirements: AutoUpdate success ✅ 18.3s
WARNING ⚠️ requireme